In [2]:
"""
Merge the real labeling sheet and synthetic negatives sheet into one
combined, shuffled file.

- Adds a 'source' column ('real' or 'synthetic') so provenance is never lost,
  even after shuffling.
- Handles the fact that the synthetic sheet has no wos_id/doi columns
  (fills them blank) without breaking the merge.
- Shuffle uses a fixed random seed, so the result is reproducible if you
  rerun this script (same shuffle order every time, not different each run).

Output: merged_shuffled.xlsx
"""

import pandas as pd
import numpy as np

REAL_PATH = "step1_labeling_sheet.xlsx"
SYNTHETIC_PATH = "synthetic_negatives.csv"
OUTPUT_PATH = "merged_shuffled.xlsx"
RANDOM_SEED = 42


def read_table(path: str) -> pd.DataFrame:
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1252")


def main():
    real = read_table(REAL_PATH)
    real["source"] = "real"

    synthetic = read_table(SYNTHETIC_PATH)
    synthetic["source"] = "synthetic"

    # Align columns: any column present in one sheet but not the other gets
    # filled with blank/NaN in the sheet that's missing it, rather than
    # erroring or silently dropping data.
    all_columns = list(dict.fromkeys(list(real.columns) + list(synthetic.columns)))
    real = real.reindex(columns=all_columns)
    synthetic = synthetic.reindex(columns=all_columns)

    combined = pd.concat([real, synthetic], ignore_index=True)

    # Shuffle with a fixed seed for reproducibility.
    rng = np.random.default_rng(RANDOM_SEED)
    shuffled_order = rng.permutation(len(combined))
    combined = combined.iloc[shuffled_order].reset_index(drop=True)

    combined.to_excel(OUTPUT_PATH, index=False)

    print(f"Real rows: {len(real)}")
    print(f"Synthetic rows: {len(synthetic)}")
    print(f"Combined + shuffled: {len(combined)} rows -> {OUTPUT_PATH}")
    print(f"Columns: {list(combined.columns)}")


if __name__ == "__main__":
    main()

Real rows: 300
Synthetic rows: 133
Combined + shuffled: 433 rows -> merged_shuffled.xlsx
Columns: ['wos_id', 'doi', 'title', 'abstract', 'label', 'source_type', 'source_specific', 'isolate_concentrate_flour', 'modification_step', 'application_property', 'trap_type', 'reason', 'source']
